# SQL Embedding Generation using BERT (Masked Language Modeling)

This notebook fine-tunes a pre-trained BERT model on our SQL query dataset. BERT's bidirectional attention is highly effective at capturing the context of SQL tokens, providing much faster training and potentially better embeddings than a custom Tree-LSTM.

In [1]:
import torch
from transformers import BertTokenizer, BertForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from torch.utils.data import Dataset
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Preprocessed Data
We use the tokens from your existing preprocessed dataset.

In [3]:
path = "/kaggle/input/datadbms/final_encoded_data.pt"
if not os.path.exists(path):
    path = "final_encoded_data.pt"

data = torch.load(path, map_location="cpu")
encoded_queries = data["encoded_queries"] 
id2token = data["id2token"]

print(f"Loaded {len(encoded_queries)} queries.")

Loaded 859973 queries.


## 2. Prepare Dataset
BERT expects text input. We convert our token IDs back to strings so the BERT tokenizer can handle them correctly (handling its own sub-word tokenization).

In [4]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
custom_tokens = ['NUM_LITERAL', 'STR_LITERAL', 'HEX_LITERAL']
tokenizer.add_tokens(custom_tokens)
class SQLDataset(Dataset):
    def __init__(self, queries, id2token, tokenizer, max_length=128, limit=100000,set_number=0):
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # For BERT, we convert back to string first
        # This allows BERT's specific sub-word tokenizer to work
        self.texts = []
        queries_using=queries[:limit]+queries[-limit:]
        for q_ids in queries_using:
            toks = []
            for i in q_ids:
                tok_str = id2token[i]
                if tok_str in ['<PAD>', '<START>', '<END>']:
                    continue
                if tok_str == '<UNK>':#mapping unkown token to the bert unknown
                    tok_str = '[UNK]'
                toks.append(tok_str)
            self.texts.append(" ".join(toks))

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.tokenizer(
            self.texts[idx], 
            truncation=True, 
            max_length=self.max_length
        )

# Create dataset using 100k queries for efficient training
full_dataset = SQLDataset(encoded_queries, id2token, tokenizer, limit=80000)

# Split into train and val
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
print(train_size,val_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

144000 16000


## 3. Fine-Tuning with Masked LM
We use the `DataCollatorForLanguageModeling` to automatically handle masking 15% of the tokens.

In [5]:
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model.resize_token_embeddings(len(tokenizer)) 

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir="./bert_sql_results",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    save_steps=1000,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=1000,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none",
    fp16=True,                      # Mixed precision speedup
    dataloader_num_workers=4,       # Faster data loading
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Starting BERT training...")
trainer.train(resume_from_checkpoint=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'be

Starting BERT training...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=9000, training_loss=0.019777665032280817, metrics={'train_runtime': 3416.8565, 'train_samples_per_second': 168.576, 'train_steps_per_second': 2.634, 'total_flos': 3.7901494370304e+16, 'train_loss': 0.019777665032280817, 'epoch': 4.0})

## 4. Extracting Embeddings
Once trained, we can get the embedding for any query by taking the output of the [CLS] token from the base BERT model.

In [6]:
from transformers import BertTokenizer, BertForMaskedLM
model_path = "/kaggle/working/bert_sql_results/checkpoint-9000" 

print("Loading the best fine-tuned model and tokenizer...")

tokenizer = BertTokenizer.from_pretrained(model_path)

model = BertForMaskedLM.from_pretrained(model_path)

model.to(device)
model.eval()


Loading the best fine-tuned model and tokenizer...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30525, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [7]:
test_str = "SELECT * FROM table WHERE col = NUM_LITERAL"
tokens = tokenizer.tokenize(test_str)
print(f"Tokenization check: {tokens}")

Tokenization check: ['select', '*', 'from', 'table', 'where', 'col', '=', 'num_literal']


In [8]:
import random
import torch
import numpy as np

def get_cls_embeddings(model, tokenizer, query_list):
    """
    Extracts [CLS] token embeddings for a list of SQL queries using the fine-tuned BERT model.
    """
    model.eval()
    model.to(device)
    
    inputs = tokenizer(
        query_list, 
        return_tensors='pt', 
        padding=True, 
        truncation=True, 
        max_length=128
    ).to(device)
    
    with torch.no_grad():
        outputs = model.bert(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :]
        
    return embeddings.cpu().numpy()

random_indices = random.sample(range(len(full_dataset)), 3)
sample_texts = [full_dataset.texts[i] for i in random_indices]
vectors = get_cls_embeddings(model, tokenizer, sample_texts)

for i, text in enumerate(sample_texts):
    print(f"Index: {random_indices[i]}")
    print(f"SQL: {text}")
    print(f"Vector (First 5 dims): {vectors[i][:5]}")
    print("-" * 30)

Index: 29184
SQL: select description from dbobjects where name = STR_LITERAL
Vector (First 5 dims): [-0.8436236   0.09599642 -0.8083388  -0.13409317 -0.09078841]
------------------------------
Index: 6556
SQL: select p . ra , p . dec , p . htmid , q . objid , q . photoz , q . photozerr , q . flag from photoobjall p , photoz2 q where ( ( p . htmid >= NUM_LITERAL and p . htmid <= NUM_LITERAL ) ) and p . objid = q . objid
Vector (First 5 dims): [-0.6926419  -1.0224559  -0.65130824  0.46544504  0.04396652]
------------------------------
Index: 72097
SQL: select ra , dec , petromag_u , petromag_g , petromag_r , petromag_i , petromag_z , petror50_r , probpsf from phototag p , fgetobjfromrecteq ( NUM_LITERAL , NUM_LITERAL , NUM_LITERAL , NUM_LITERAL ) r where p . objid = r . objid
Vector (First 5 dims): [-1.1115304  -0.6891766   0.02512879 -0.10472898  0.14938296]
------------------------------


In [9]:
import os
import shutil
!rm -rf /kaggle/working/bert_query2vec_model
def export_model_package(model, tokenizer, output_dir):
    """
    Saves the model and tokenizer to a local directory and compresses them into a single zip archive.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    shutil.make_archive(output_dir, 'zip', output_dir)
    
    return f"{output_dir}.zip"

save_name = "bert_query2vec_model"
zip_file_path = export_model_package(model, tokenizer, save_name)
print(f"Exported model to: {zip_file_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Exported model to: bert_query2vec_model.zip
